<a href="https://colab.research.google.com/github/acapodanno/agent-ai/blob/main/agent_qa_stm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Esercitazione — Agente QA con Memoria STM

## Obiettivo

Costruire un agente QA con LangChain dotato di memoria a breve
termine, confrontare due strategie di gestione della history
e osservare concretamente cosa arriva al modello ad ogni turno.

---

## Contesto

Un agente QA deve mantenere il filo di una conversazione lunga:
ricordare risposte già date, riferirsi a informazioni precedenti
e comportarsi in modo coerente anche dopo molti turni.

Esempio atteso:
- Turno 3: "Il Python è ottimo per il machine learning."
- Turno 9: "Come dicevo prima, Python è la scelta giusta per ML."
- Turno 15: "Ricordi cosa ti ho detto su Python?" → l'agente
  deve rispondere correttamente anche con buffer fisso

---

## Parte 1 — Buffer fisso con ConversationBufferMemory

### Requisiti

1. Costruisci una chain con `ConversationBufferMemory` e `k=5`
2. Conduci una conversazione di **almeno 15 turni** sullo stesso
   argomento (es. consigli su linguaggi di programmazione,
   scelta di un laptop, pianificazione di un viaggio)
3. Al turno 8 e al turno 14 fai una domanda che richiede
   di ricordare qualcosa detto nel turno 2:
   - Turno 2: introduci un'informazione specifica
     (es. "Ho un budget di 1.200€")
   - Turno 8: "Ricordi il mio budget?"
   - Turno 14: stessa domanda — con `k=5` la risposta
     sarà diversa. Perché?
4. Logga manualmente i messaggi in memoria ad ogni turno:
```python
   print(memory.chat_memory.messages)
```

### Output atteso

Una tabella con:

| Turno | Domanda | Risposta | Messaggi in memoria |
|---|---|---|---|
| 2 | "Ho un budget di 1.200€..." | ... | 2 |
| 8 | "Ricordi il mio budget?" | ... | 5 |
| 14 | "Ricordi il mio budget?" | ... | 5 |

---

## Parte 2 — Summarization con ConversationSummaryMemory

### Requisiti

1. Reimplementa la stessa conversazione con
   `ConversationSummaryMemory` usando `gpt-4o-mini`
   come modello per il summary
2. Stessa conversazione di 15 turni, stesse domande
   ai turni 8 e 14
3. Logga il summary corrente ad ogni turno:
```python
   print(memory.moving_summary_buffer)
```
4. Verifica che il summary al turno 14 contenga ancora
   l'informazione del turno 2 (il budget)

### Output atteso

Lo stesso formato tabellare della Parte 1, con una colonna
aggiuntiva per il summary corrente.

---

## Parte 3 — Verifica con LangSmith o log manuali

### Opzione A — LangSmith (consigliato)

Configura LangSmith con le variabili d'ambiente:
```python
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = "..."
os.environ["LANGSMITH_PROJECT"] = "esercitazione-stm"
```

Per ogni strategia, apri il trace in LangSmith e cattura
uno screenshot che mostri:
- I messaggi esatti inviati al modello
- Il numero di token usati per turno

### Opzione B — Log manuali

Se non hai LangSmith, usa un callback:
```python
from langchain_core.callbacks.base import BaseCallbackHandler

class TokenLogger(BaseCallbackHandler):
    def on_llm_start(self, serialized, prompts, **kwargs):
        import tiktoken
        enc = tiktoken.encoding_for_model("gpt-4o")
        tokens = len(enc.encode(prompts[0]))
        print(f"Token inviati al modello: {tokens}")
```

---

## Parte 4 — Confronto finale

Compila questa tabella con i dati reali della tua esecuzione:

| Metrica | Buffer fisso (k=5) | Summarization |
|---|---|---|
| Token medi per turno | | |
| Ricorda info del turno 2 al turno 8? | | |
| Ricorda info del turno 2 al turno 14? | | |
| Costo stimato su 100 turni | | |

Scrivi **100 parole** su quando useresti una strategia
rispetto all'altra.

---

## Criteri di valutazione

| Criterio | Peso |
|---|---|
| Conversazione funzionante con entrambe le strategie | 30% |
| Log corretti dei messaggi in memoria | 25% |
| Verifica con LangSmith o callback | 25% |
| Confronto finale con dati reali | 20% |

---

## Consegna

- `esercitazione_stm.ipynb`
- Screenshot LangSmith (o output del TokenLogger)

## ⚠️ Nota sulla deprecazione

`ConversationBufferWindowMemory` è **deprecata** a partire da
LangChain v0.3.1. La mostriamo perché la troverai in moltissimi
tutorial e repository esistenti — è importante saperla leggere.

Per nuovi progetti, usa `InMemorySaver` di LangGraph con
`trim_messages` — che vedremo nei moduli successivi.
